# Study Pyvis

In [1]:
from openpyxl.styles.builtins import title
from pyvis.network import Network

net = Network(notebook=True)

In [2]:
net.add_node(1, label="Node A", shape="circle")
net.add_node(2, label="Node B", shape="square")
net.add_node(3, label="Node C", shape="database")

In [3]:
net.get_node(1)

{'color': '#97c2fc', 'id': 1, 'label': 'Node A', 'shape': 'circle'}

In [4]:
net.add_nodes(
    [4, 5, 6],
    value=[-2, -3, -4],
    title=["Node 4 is here", "Node 5 there", "Node 6 acolá"],
    x=[21.4, 54.2, 11.2],
    y=[100.2, 23.54, 32.1],
    label=["Node 4", "Node 5", "Node 6"],
    color=['#00ff1e', '#162347', '#dd4b39']
)

In [10]:
net.toggle_physics(True)
net.show('mygraph.html')

mygraph.html


In [11]:
import torch
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.nn import GCNConv, dense_diff_pool
from torch_geometric.utils import to_dense_adj, to_networkx

import networkx as nx
from pyvis.network import Network

# 1. Prepare a small graph
dataset = TUDataset(root='/tmp/TUD', name='MUTAG')
data = dataset[0]

x = data.x
if x is None:
    x = torch.eye(data.num_nodes)
adj = to_dense_adj(data.edge_index)[0]  # shape [N, N]


# 2. Define a GNN + 2-layer DiffPool architecture

class TwoLevelDiffPool(torch.nn.Module):
    def __init__(self, in_channels, hidden, num_clusters1, num_clusters2):
        super(TwoLevelDiffPool, self).__init__()
        # Embedding path for level 1
        self.embed1 = GCNConv(in_channels, hidden)
        # Assignment path level 1
        self.pool1 = GCNConv(in_channels, hidden)
        self.assign_lin1 = torch.nn.Linear(hidden, num_clusters1)

        # Embedding path for level 2 (on coarsened graph from level 1)
        self.embed2 = GCNConv(hidden, hidden)
        # Assignment path level 2
        self.pool2 = GCNConv(hidden, hidden)
        self.assign_lin2 = torch.nn.Linear(hidden, num_clusters2)

    def forward(self, x, edge_index, adj_dense):
        # --- Level 1 ---
        # Embeddings
        z1 = F.relu(self.embed1(x, edge_index))  # [N, hidden]
        # Assignment
        s1 = F.relu(self.pool1(x, edge_index))  # [N, hidden]
        s1 = self.assign_lin1(s1)  # [N, C1]
        s1 = F.softmax(s1, dim=-1)  # soft assignments to clusters 1

        # Perform pooling 1
        x1, adj1, link_loss1, ent_loss1 = dense_diff_pool(
            z1.unsqueeze(0), adj_dense.unsqueeze(0), s1.unsqueeze(0)
        )
        # Remove batch dim
        x1 = x1.squeeze(0)
        adj1 = adj1.squeeze(0)

        # Build edge_index1 for next layer from adj1 (dense → sparse)
        # (We’ll convert to edge_index via threshold / nonzero)
        edge_index1 = (adj1 > 1e-6).nonzero(as_tuple=False).t().contiguous()

        # --- Level 2 ---
        # Embedding on pooled features
        z2 = F.relu(self.embed2(x1, edge_index1))
        # Assignment level 2
        s2 = F.relu(self.pool2(x1, edge_index1))
        s2 = self.assign_lin2(s2)  # [C1, C2]
        s2 = F.softmax(s2, dim=-1)

        # Perform pooling 2
        x2, adj2, link_loss2, ent_loss2 = dense_diff_pool(
            z2.unsqueeze(0), adj1.unsqueeze(0), s2.unsqueeze(0)
        )
        x2 = x2.squeeze(0)  # [C2, hidden]
        adj2 = adj2.squeeze(0)  # [C2, C2]

        return {
            'z0': x,  # original features (just for consistency)
            'adj0': adj_dense,
            's1': s1,
            'x1': x1,
            'adj1': adj1,
            's2': s2,
            'x2': x2,
            'adj2': adj2
        }


# 3. Instantiate with cluster sizes
N = data.num_nodes
# Example: reduce to ~ half, then to 1
C1 = max(2, N // 2)
C2 = 1  # final single cluster
model = TwoLevelDiffPool(in_channels=x.size(1), hidden=32, num_clusters1=C1, num_clusters2=C2)

# Forward pass
out = model(x, data.edge_index, adj)

# Extract assignments and coarsened graphs
s1 = out['s1'].detach().cpu()  # [N, C1]
s2 = out['s2'].detach().cpu()  # [C1, 1]
adj0 = out['adj0'].detach().cpu().numpy()
adj1 = out['adj1'].detach().cpu().numpy()
adj2 = out['adj2'].detach().cpu().numpy()

# Derive **hard assignments** (argmax) at each level
assign1 = torch.argmax(s1, dim=-1).numpy()  # each original node → cluster in level1
# For level2: cluster in level1 → cluster in level2 (but since C2=1, it's trivial)
assign2 = torch.argmax(s2, dim=-1).numpy()  # shape [C1,]

# Build NetworkX graphs for visualization

# Original graph with cluster1 membership as attribute
G0 = to_networkx(data, to_undirected=True)
for i in G0.nodes():
    G0.nodes[i]['cluster1'] = int(assign1[i])

# Pooled graph level1: nodes are clusters 1
G1 = nx.Graph()
C1_count = adj1.shape[0]
for c in range(C1_count):
    G1.add_node(f"c1_{c}", cluster_lvl=1, cluster1_id=c)
for i in range(C1_count):
    for j in range(C1_count):
        w = adj1[i, j]
        if w > 1e-6:
            G1.add_edge(f"c1_{i}", f"c1_{j}", weight=w)

# Also record which cluster1 → cluster2 mapping
# So in G1 we can annotate which cluster2 each cluster1 belongs to:
for node in G1.nodes(data=True):
    c1 = node[1]['cluster1_id']
    c2 = assign2[c1]
    node[1]['mapped_to_lvl2'] = int(c2)


# 4. Visualization function (with Pyvis) that can show levels clearly
def visualize(G, title, color_attr=None, size_attr=None, filename="graph.html"):
    net = Network(notebook=True, height="600px", width="800px", bgcolor="#ffffff")
    net.force_atlas_2based()

    for n, d in G.nodes(data=True):
        color = "#97c2fc"
        title_txt = str(n)
        if color_attr is not None and color_attr in d:
            v = d[color_attr]
            # map to a color (for simplicity, cycle through a small palette)
            palette = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'cyan']
            color = palette[v % len(palette)]
            title_txt += f"<br>{color_attr} = {v}"
        net.add_node(n, label=str(n), title=title_txt, color=color)

    for u, v, d in G.edges(data=True):
        # u, v also must be int or str
        u_id = u if isinstance(u, (int, str)) else str(u)
        v_id = v if isinstance(v, (int, str)) else str(v)

        w = d.get('weight', d.get('value', 1.0))
        # Cast weight to float
        try:
            w_f = float(w)
        except Exception:
            w_f = float(w.item()) if hasattr(w, 'item') else float(w)
        net.add_edge(u_id, v_id, value=w_f)

    net.show(filename)
    print(f" Saved: {title} → {filename}")


# Visualize all levels
visualize(G0, title="Original + level1 clusters", color_attr='cluster1', filename="level0.html")
visualize(G1, title="Level1 pooled graph", color_attr='mapped_to_lvl2', filename="level1.html")
visualize(G2, title="Level2 pooled (final)", filename="level2.html")


level0.html
 Saved: Original + level1 clusters → level0.html
level1.html
 Saved: Level1 pooled graph → level1.html
level2.html
 Saved: Level2 pooled (final) → level2.html


# Game of Thrones

In [57]:
from pyvis.network import Network
import pandas as pd

In [58]:
import glob
import pandas as pd

got_net = Network(height="100vh", width="100%", bgcolor="#222222", font_color="white", notebook=False, select_menu=True,
                  filter_menu=True)

got_net.barnes_hut()

dfs = []
fs = glob.glob("got/*.csv")
for f in fs:
    df = pd.read_csv(f)
    dfs.append(df)

got_data = pd.concat(dfs[:1])
got_data.head()

,Source,Target,Type,weight,book
0,Addam-Marbrand,Jaime-Lannister,Undirected,3,1
1,Addam-Marbrand,Tywin-Lannister,Undirected,6,1
2,Aegon-I-Targaryen,Daenerys-Targaryen,Undirected,5,1
3,Aegon-I-Targaryen,Eddard-Stark,Undirected,4,1
4,Aemon-Targaryen-(Maester-Aemon),Alliser-Thorne,Undirected,4,1


In [59]:
sources = got_data['Source']
targets = got_data["Target"]
weights = got_data["weight"]

In [60]:
edge_data = zip(sources, targets, weights)

for e in edge_data:
    src = e[0]
    tgt = e[1]
    weight = e[2]

    got_net.add_node(src, src, title=src)
    got_net.add_node(tgt, tgt, title=tgt)
    got_net.add_edge(src, tgt, value=weight, label=weight)

neighbor_map = got_net.get_adj_list()

for node in got_net.nodes:
    neighbors = list(neighbor_map[node["id"]])

    node["title"] += f" Neighbors ({len(neighbors)}):\n"
    if len(neighbors) > 5:
        node["title"] += "\n".join(neighbors[:5]) + "\n..."
    else:
        node["title"] += "\n".join(neighbors)

    if "value" in node:
        node["value"] += len(neighbor_map[node["id"]])
    else:
        node["value"] = len(neighbor_map[node["id"]])

In [61]:
got_net.show_buttons(filter_=['physics'])

got_net.show("gameofthrones.html", notebook=False)

gameofthrones.html
